# FastAPI Serving Benchmark

Custom Python benchmarking for FastAPI-served models.

Two modes:
1. **Sequential requests** -- 100 requests one at a time
2. **Concurrent requests** -- 1000 parallel requests via ThreadPoolExecutor

**Metrics**: median latency, p95, p99, throughput (req/sec)

In [ ]:
import requests
import time
import numpy as np
import pandas as pd
import concurrent.futures

FASTAPI_URL = "http://fastapi_server:8000/predict"

payload = {
    "transaction_id": "txn_001",
    "payee": "TESCO STORES 3456",
    "amount": -45.20,
    "date": "2024-03-15",
}

all_benchmark_results = []

## Verify endpoint is reachable

In [ ]:
health = requests.get("http://fastapi_server:8000/health")
print(f"Health check: {health.json()}")

test_response = requests.post(FASTAPI_URL, json=payload)
print(f"Test prediction: {test_response.json()}")

## 1. Sequential Requests (100 requests)

In [ ]:
num_requests = 100
inference_times = []

for _ in range(num_requests):
    start_time = time.time()
    response = requests.post(FASTAPI_URL, json=payload)
    end_time = time.time()
    if response.status_code == 200:
        inference_times.append(end_time - start_time)
    else:
        print(f"Error: {response.status_code}, Response: {response.text}")

inference_times = np.array(inference_times)
median_time = np.median(inference_times)
percentile_95 = np.percentile(inference_times, 95)
percentile_99 = np.percentile(inference_times, 99)
throughput = num_requests / inference_times.sum()

print(f"=== Sequential Requests ({num_requests}) ===")
print(f"Median latency:   {1000*median_time:.2f} ms")
print(f"95th percentile:  {1000*percentile_95:.2f} ms")
print(f"99th percentile:  {1000*percentile_99:.2f} ms")
print(f"Throughput:       {throughput:.2f} req/sec")

all_benchmark_results.append({
    "mode": "sequential",
    "num_requests": num_requests,
    "median_latency_ms": round(1000*median_time, 2),
    "p95_latency_ms": round(1000*percentile_95, 2),
    "p99_latency_ms": round(1000*percentile_99, 2),
    "throughput_rps": round(throughput, 2),
})

## 2. Concurrent Requests (1000 requests, 16 workers)

In [ ]:
def send_request(payload):
    start_time = time.time()
    response = requests.post(FASTAPI_URL, json=payload)
    end_time = time.time()
    if response.status_code == 200:
        return end_time - start_time
    else:
        print(f"Error: {response.status_code}, Response: {response.text}")
        return None

def run_concurrent_tests(num_requests, payload, max_workers=16):
    inference_times = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(send_request, payload) for _ in range(num_requests)]
        for future in concurrent.futures.as_completed(futures):
            result = future.result()
            if result is not None:
                inference_times.append(result)
    return inference_times

num_requests_concurrent = 1000
start_time = time.time()
concurrent_times = run_concurrent_tests(num_requests_concurrent, payload, max_workers=16)
total_time = time.time() - start_time

concurrent_times = np.array(concurrent_times)
median_time = np.median(concurrent_times)
percentile_95 = np.percentile(concurrent_times, 95)
percentile_99 = np.percentile(concurrent_times, 99)
throughput = num_requests_concurrent / total_time

print(f"=== Concurrent Requests ({num_requests_concurrent}, 16 workers) ===")
print(f"Median latency:   {1000*median_time:.2f} ms")
print(f"95th percentile:  {1000*percentile_95:.2f} ms")
print(f"99th percentile:  {1000*percentile_99:.2f} ms")
print(f"Throughput:       {throughput:.2f} req/sec")
print(f"Total wall time:  {total_time:.2f} s")

all_benchmark_results.append({
    "mode": "concurrent_16_workers",
    "num_requests": num_requests_concurrent,
    "median_latency_ms": round(1000*median_time, 2),
    "p95_latency_ms": round(1000*percentile_95, 2),
    "p99_latency_ms": round(1000*percentile_99, 2),
    "throughput_rps": round(throughput, 2),
})

## 3. Concurrent Requests with varying worker counts

In [ ]:
worker_counts = [1, 4, 8, 16, 32]

for workers in worker_counts:
    start = time.time()
    times = run_concurrent_tests(500, payload, max_workers=workers)
    total = time.time() - start
    times = np.array(times)

    result = {
        "mode": f"concurrent_{workers}_workers",
        "num_requests": 500,
        "median_latency_ms": round(1000*np.median(times), 2),
        "p95_latency_ms": round(1000*np.percentile(times, 95), 2),
        "p99_latency_ms": round(1000*np.percentile(times, 99), 2),
        "throughput_rps": round(500 / total, 2),
    }
    all_benchmark_results.append(result)
    print(f"Workers={workers}: throughput={result['throughput_rps']:.1f} rps, "
          f"p50={result['median_latency_ms']:.1f}ms, p99={result['p99_latency_ms']:.1f}ms")

## Summary

In [ ]:
results_df = pd.DataFrame(all_benchmark_results)
print(results_df.to_string(index=False))
results_df.to_csv("results/fastapi_benchmark.csv", index=False)
print("\nResults saved to results/fastapi_benchmark.csv")